In [ ]:
# %pip install -qU \
#   "langchain>=0.3" \
#   "langgraph>=0.2" \
#   "langchain-google-genai>=2.0" \
#   "google-genai>=1.0" \
#   "langchain-mcp-adapters==0.2.1" \
#   "nest_asyncio"

In [ ]:
!node --version
!npx --version

In [2]:
import os
from pathlib import Path

WORKDIR = str((Path.cwd() / "sandbox").resolve())
os.makedirs(WORKDIR, exist_ok=True)

In [ ]:
# %pip install -qU mcp-server-git

In [3]:
import nest_asyncio, shutil, sys
nest_asyncio.apply()

NPX = shutil.which("npx.cmd") or shutil.which("npx")

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

In [5]:
import asyncio, os, sys, threading

# 1) stderr des serveurs -> un vrai fichier (sinon .fileno() explose dans Jupyter)
import langchain_mcp_adapters.sessions as _sess
_LOG = open("mcp_servers.log", "a", encoding="utf-8")
_orig_stdio = _sess.stdio_client
_sess.stdio_client = lambda server, errlog=_LOG: _orig_stdio(server, errlog=errlog)

# 2) exécuter les coroutines MCP dans un thread muni d'une ProactorEventLoop
def run_async(coro):
    box = {}
    def worker():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        try: box["v"] = loop.run_until_complete(coro)
        except BaseException as e: box["e"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "e" in box: raise box["e"]
    return box["v"]

In [6]:
tools = run_async(client.get_tools())
for t in tools:
    print(t.name)

filesystem_read_file
filesystem_read_text_file
filesystem_read_media_file
filesystem_read_multiple_files
filesystem_write_file
filesystem_edit_file
filesystem_create_directory
filesystem_list_directory
filesystem_list_directory_with_sizes
filesystem_directory_tree
filesystem_move_file
filesystem_search_files
filesystem_get_file_info
filesystem_list_allowed_directories
git_git_status
git_git_diff_unstaged
git_git_diff_staged
git_git_diff
git_git_commit
git_git_add
git_git_reset
git_git_log
git_git_create_branch
git_git_checkout
git_git_show
git_git_branch


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

POLICY = """You are a dev assistant working on a local git repository through MCP tools.

How to work:
- Inspect before acting: read the relevant files before editing them.
- Never guess a fix. Run the tests to confirm a failure, then again to confirm the fix.
- Make the smallest possible edit.
- If the user asks for a read-only report, do NOT call any tool that writes, edits or commits.
- Commit only when explicitly asked, with a Conventional Commits message.
- End with a short summary: what you changed and the evidence it works.
"""

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
# couldn't use ChatGoogleGenerativeAI, so I used another alternative with ollama
llm = ChatOllama(model="qwen3.5:4b", temperature=0)

agent = create_agent(llm, tools, system_prompt=POLICY)

In [ ]:
# %pip install -qU "fastmcp>=2.0.0"

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.21.0 requires fsspec[http]<=2024.6.1,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.


In [ ]:
"""Custom MCP server: test runner / linter for the sandbox repo."""
import subprocess, sys
from pathlib import Path
# from fastmcp import FastMCP
from mcp.server.fastmcp import FastMCP

REPO = Path(__file__).parent / "sandbox"
mcp = FastMCP(name="devtools")

def _run(args, timeout=120):
    p = subprocess.run(args, cwd=REPO, capture_output=True, text=True, timeout=timeout)
    return p.returncode, (p.stdout + p.stderr).strip()[-4000:]

@mcp.tool()
def ping() -> str:
    """Health check tool."""
    return "pong"

@mcp.tool()
def run_pytest(test_path: str = "") -> str:
    """Run the project's pytest suite. Optionally restrict to one file or test id."""
    args = [sys.executable, "-m", "pytest", "-q", "--no-header"]
    if test_path:
        args.append(test_path)
    rc, out = _run(args)
    return f"exit_code={rc} ({'PASS' if rc == 0 else 'FAIL'})\n\n{out}"

@mcp.tool()
def run_ruff(fix: bool = False) -> str:
    """Lint the project with ruff. Set fix=True to auto-apply safe fixes."""
    args = [sys.executable, "-m", "ruff", "check", "."]
    if fix:
        args.append("--fix")
    rc, out = _run(args)
    return f"exit_code={rc}\n\n{out or 'no issues found'}"

@mcp.tool()
def project_health() -> str:
    """Read-only summary of the repo: file count, test status, lint status. Modifies nothing."""
    py = sorted(p.relative_to(REPO).as_posix() for p in REPO.rglob("*.py") if ".git" not in p.parts)
    rc_t, out_t = _run([sys.executable, "-m", "pytest", "-q", "--no-header"])
    rc_l, _ = _run([sys.executable, "-m", "ruff", "check", "."])
    return (f"python_files={len(py)}: {py}\n"
            f"tests={'PASS' if rc_t == 0 else 'FAIL'}\n"
            f"lint={'CLEAN' if rc_l == 0 else 'ISSUES'}\n"
            f"last_test_output:\n{out_t[-800:]}")

if __name__ == "__main__":
    mcp.run(transport="stdio")


In [10]:
mcp_connections["devtools"] = {
    "transport": "stdio",
    "command": sys.executable,
    "args": [str((Path.cwd() / "devtools_server.py").resolve())],
    "env": {**os.environ},
}

client2 = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
tools2 = run_async(client2.get_tools())

print("Tool count:", len(tools2))
print([t.name for t in tools2 if t.name.startswith("devtools")])

Tool count: 30
['devtools_ping', 'devtools_run_pytest', 'devtools_run_ruff', 'devtools_project_health']
